# Patent phrase matching

Chirag Duhoon

Cross-encoder regression, a frozen sentence-embedding baseline, two controlled experiments and error analysis.

Adapted from the course [starter and project brief](https://github.com/bk-anupam/generative_ai_shared/tree/master/module3/labs/patent_phrase_matching).

Use the original Kaggle competition **train.csv** and **test.csv**, not the later fully labelled release. Attach the competition data and the saved checkpoints from `download_weights_chirag.ipynb`. Run with a T4/P100 and internet off. The experiment cell trains three models sequentially. It writes measurements and a four-page report, then selects by validation correlation. Leaderboard scores and a screenshot require a real late submission from the Kaggle account; they are not generated here.

In [ ]:
import gc
import glob
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset, Value
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
VAL_FRACTION = 0.2
MAX_LEN = 64
EPOCHS = 3
BATCH_SIZE = 32
MODEL_ID = "microsoft/deberta-v3-small"
EMBED_ID = "sentence-transformers/all-MiniLM-L6-v2"
OUT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('patent_outputs')
OUT.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
transformers.set_seed(SEED)
print('torch', torch.__version__, '| transformers', transformers.__version__, '| device', device)

def checkpoint(model_id):
    short = model_id.rsplit('/', 1)[-1]
    hits = sorted(Path('/kaggle/input').glob(f'**/{short}/config.json'))
    return str(hits[0].parent) if hits else model_id

model_path, embed_path = checkpoint(MODEL_ID), checkpoint(EMBED_ID)
print('cross-encoder:', model_path, '| bi-encoder:', embed_path)

In [ ]:
candidates = sorted(Path('/kaggle/input').glob('**/us-patent-phrase-to-phrase-matching/train.csv'))
if not candidates:
    local_data = Path(os.environ.get('PATENT_DATA_DIR', 'patent_data'))
    candidates = [local_data / 'train.csv'] if (local_data / 'train.csv').exists() else []
if not candidates:
    raise FileNotFoundError('Attach the original competition data, or set PATENT_DATA_DIR to its directory.')
data_dir = candidates[0].parent
data = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
assert len(data) == 36473 and data.anchor.nunique() == 733, 'Expected the original competition training split'
assert 'score' not in test, 'Test labels must never be loaded'
assert data.id.is_unique and test.id.is_unique
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(data, groups=data.anchor))
trn, val = data.iloc[train_idx].reset_index(drop=True), data.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
pd.DataFrame({'id': data.id, 'split': np.where(data.index.isin(val_idx), 'validation', 'train')}).to_csv(OUT / 'split.csv', index=False)
print('training:', len(trn), '| validation:', len(val), '| held-out anchors:', val.anchor.nunique())
print(data.score.value_counts().sort_index())

Whole anchors are held out because many targets share an anchor. A row split would let the model see the same anchor during training and validation, making unseen-anchor performance look better than it is. The split and seed stay fixed for the frozen baseline and all three trained runs.

The frozen bi-encoder represents each phrase independently, then compares two vectors using cosine similarity. The cross-encoder reads both phrases in one input, permitting token-level interaction, and learns the regression task from training pairs. Differences between them reflect both the model architecture and task-specific training, so this comparison cannot isolate the effect of fine-tuning alone.

In [ ]:
def pearson(pred, target):
    pred, target = np.asarray(pred), np.asarray(target)
    if len(pred) < 2 or pred.std() == 0 or target.std() == 0:
        return float('nan')
    return float(np.corrcoef(pred, target)[0, 1])

@torch.no_grad()
def sentence_embeddings(texts, model, tokenizer, batch=128):
    pieces = []
    model.eval()
    for start in range(0, len(texts), batch):
        enc = tokenizer(texts[start:start+batch], padding=True, truncation=True,
                        max_length=256, return_tensors='pt').to(device)
        hidden = model(**enc).last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)
        pieces.append(torch.nn.functional.normalize(mean, dim=-1).cpu())
    return torch.cat(pieces)

start = time.perf_counter()
embed_tok = AutoTokenizer.from_pretrained(embed_path)
embed_model = AutoModel.from_pretrained(embed_path).to(device)
phrases = sorted(set(val.anchor) | set(val.target))
vectors = sentence_embeddings(phrases, embed_model, embed_tok)
lookup = {phrase: i for i, phrase in enumerate(phrases)}
a = vectors[[lookup[s] for s in val.anchor]]
b = vectors[[lookup[s] for s in val.target]]
bi_preds = (a * b).sum(-1).numpy()
results = [{'run': 'frozen bi-encoder', 'context': 'none', 'lr': 0,
            'pearson': pearson(bi_preds, val.score), 'training_seconds': 0,
            'inference_seconds': time.perf_counter() - start, 'trainable_parameters': 0,
            'peak_gpu_gb': np.nan}]
print(pd.DataFrame(results).to_string(index=False))
del embed_model, vectors, a, b
gc.collect()
if device == 'cuda': torch.cuda.empty_cache()

In [ ]:
CPC_SECTIONS = {
    'A': 'human necessities', 'B': 'performing operations; transporting',
    'C': 'chemistry; metallurgy', 'D': 'textiles; paper', 'E': 'fixed constructions',
    'F': 'mechanical engineering; lighting; heating; weapons; blasting',
    'G': 'physics', 'H': 'electricity'}
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single='[CLS] $A [SEP]', pair='[CLS] $A [SEP] $B:1 [SEP]:1',
    special_tokens=[('[CLS]', tokenizer.cls_token_id), ('[SEP]', tokenizer.sep_token_id)])

def to_dataset(frame, context='section', labels=True):
    second = frame.target.copy()
    if context == 'section':
        second += '; ' + frame.context.str[0].map(CPC_SECTIONS)
    elif context != 'none':
        raise ValueError(context)
    encoded = tokenizer(frame.anchor.tolist(), second.tolist(), truncation=True, max_length=MAX_LEN)
    if labels:
        encoded['labels'] = frame.score.astype('float32').tolist()
    dataset = Dataset.from_dict(encoded)
    return dataset.cast_column('labels', Value('float32')) if labels else dataset

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {'pearson': pearson(preds.reshape(-1), labels)}

print(tokenizer.decode(to_dataset(trn)[0]['input_ids']))

In [ ]:
# Each experiment changes exactly one setting from the starter baseline.
experiments = [('starter', 'section', 2e-5), ('no_context', 'none', 2e-5),
               ('lower_learning_rate', 'section', 1e-5)]
best_score, best_name, best_val, best_test = -np.inf, None, None, None
for name, context, lr in experiments:
    transformers.set_seed(SEED)
    ds_train, ds_val = to_dataset(trn, context), to_dataset(val, context)
    ds_test = to_dataset(test, context, labels=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=1, dtype=torch.float32)
    args = TrainingArguments(output_dir=str(OUT / ('trainer_' + name)), num_train_epochs=EPOCHS,
        learning_rate=lr, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=2*BATCH_SIZE, warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='no', logging_steps=100,
        fp16=device == 'cuda', report_to='none', seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
    if device == 'cuda': torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer.train()
    seconds = time.perf_counter() - start
    pred = trainer.predict(ds_val).predictions.reshape(-1)
    score = pearson(pred, val.score)
    results.append({'run': name, 'context': context, 'lr': lr, 'pearson': score,
                    'training_seconds': seconds, 'inference_seconds': np.nan,
                    'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
                    'peak_gpu_gb': torch.cuda.max_memory_allocated()/1e9 if device == 'cuda' else np.nan})
    saved = val[['id','anchor','target','context','score']].copy()
    saved['pred'] = pred
    saved.to_csv(OUT / f'val_predictions_{name}.csv', index=False)
    pd.DataFrame(trainer.state.log_history).to_csv(OUT / f'history_{name}.csv', index=False)
    if score > best_score:
        best_score, best_name, best_val = score, name, saved
        best_test = trainer.predict(ds_test).predictions.reshape(-1)
    del trainer, model
    gc.collect()
    if device == 'cuda': torch.cuda.empty_cache()

table = pd.DataFrame(results)
table.to_csv(OUT / 'experiments.csv', index=False)
print(table.to_string(index=False))
baseline_score = table.loc[table.run == 'starter', 'pearson'].item()
for name in ('no_context', 'lower_learning_rate'):
    delta = table.loc[table.run == name, 'pearson'].item() - baseline_score
    print(name, 'delta from starter:', round(delta, 4),
          '| too small to distinguish from seed noise' if abs(delta) <= 0.005 else '| single-seed result; replication needed')
print('Selected by validation only:', best_name, round(best_score, 4))
best_val.to_csv(OUT / 'val_predictions.csv', index=False)
(OUT / 'selected_model.json').write_text(json.dumps({'run':best_name, 'pearson':best_score,
    'seed':SEED, 'val_fraction':VAL_FRACTION, 'experiments':experiments}, indent=2))

In [ ]:
analysis = best_val.copy()
analysis['section'] = analysis.context.str[0]
analysis['absolute_error'] = (analysis.pred - analysis.score).abs()
sections = pd.DataFrame([{'section': s, 'rows': len(g), 'pearson': pearson(g.pred,g.score),
                          'MAE':g.absolute_error.mean()} for s,g in analysis.groupby('section')])
calibration = analysis.groupby('score').agg(rows=('pred','size'), mean_prediction=('pred','mean'),
                                           std_prediction=('pred','std')).reset_index()
worst = analysis.nlargest(10,'absolute_error').copy()

def error_group(row):
    if any(len(w) <= 3 for w in (row.anchor + ' ' + row.target).split()):
        return 'short forms / short technical terms (inspect)'
    if set(row.anchor.lower().split()) & set(row.target.lower().split()):
        return 'shared words with different scope (inspect)'
    return 'technical synonymy or field meaning (inspect)'

worst['inspection_group'] = worst.apply(error_group,axis=1)
sections.to_csv(OUT / 'section_metrics.csv',index=False)
calibration.to_csv(OUT / 'score_calibration.csv',index=False)
worst.to_csv(OUT / 'largest_errors.csv',index=False)
print(sections.to_string(index=False))
print(calibration.to_string(index=False))
print(worst[['anchor','target','context','score','pred','absolute_error','inspection_group']].to_string(index=False))
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(calibration.score, calibration.mean_prediction, 'o-', label='mean prediction')
ax.plot([0,1],[0,1],'--',color='gray',label='ideal calibration')
ax.set(xlabel='True score',ylabel='Mean prediction',title='Validation calibration')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'calibration.png',dpi=140); plt.show()
print('Lowest section correlations:', sections.sort_values('pearson').head(2).section.tolist())
print('Low-score mean:',calibration.iloc[0].mean_prediction,'| high-score mean:',calibration.iloc[-1].mean_prediction)

The worst-error groups above are **inspection aids**, not verified explanations. A short token can be an abbreviation or an ordinary word; shared words need not be the cause of an error. Check each pair in its CPC context before attributing a failure. Technical synonyms, differences in phrase scope, and abbreviation expansion are plausible explanations to investigate. A disputable label needs domain evidence; disagreement with this model alone does not show the label is wrong.

The per-section table includes row counts because small sections have less stable estimates. A low section correlation can reflect scarce training examples or specialized language. If the score-0 mean is above 0 and the score-1 mean below 1, predictions compress the extremes. MSE training and the greater frequency of intermediate labels can contribute. Pearson correlation is unchanged by **positive** affine rescaling, so it is not a calibration measure.

In [ ]:
submission = pd.DataFrame({'id':test.id, 'score':np.clip(best_test,0,1)})
assert len(submission) == len(test) and submission.id.equals(test.id)
assert np.isfinite(submission.score).all() and submission.score.between(0,1).all()
submission.to_csv(OUT / 'submission.csv',index=False)
print(submission.head())

# Store account-side results separately; no scores are invented.
leaderboard_path = OUT / 'leaderboard_scores.csv'
if not leaderboard_path.exists():
    pd.DataFrame(columns=['notebook_version','public_score','private_score','screenshot']).to_csv(leaderboard_path,index=False)
leaderboard = pd.read_csv(leaderboard_path)
print('Leaderboard submissions recorded:',len(leaderboard))
print('Use Kaggle Late Submission after Save & Run All; record every submission and retain the screenshot.')

In [ ]:
# Four-page report built from measured results.
import textwrap

def report_page(pdf, title, blocks):
    fig = plt.figure(figsize=(8.27,11.69))
    fig.text(.07,.95,title,fontsize=15,weight='bold',va='top')
    y = .9
    for text, size, mono in blocks:
        lines = text.splitlines() if mono else textwrap.wrap(text,100)
        fig.text(.07,y,'\n'.join(lines),fontsize=size,va='top',family='monospace' if mono else 'sans-serif')
        y -= (len(lines)*size*1.35)/842 + .025
    pdf.savefig(fig); plt.close(fig)

with PdfPages(OUT / 'patent_report_chirag.pdf') as pdf:
    report_page(pdf,'Patent phrase matching — Chirag Duhoon',[
        ('Training and validation hold out whole anchors (seed 42, validation share 0.2). The original competition test set is used only to write predictions. All experiments share the split, checkpoint, three epochs and batch size 32 except the explicitly varied factor.',10,False),
        (table[['run','pearson','training_seconds','peak_gpu_gb']].round(4).to_string(index=False),8,True),
        (f'Selected run: {best_name}; validation Pearson {best_score:.4f}. The frozen bi-encoder encodes phrases independently. The trained cross-encoder compares phrase tokens jointly. This comparison changes both architecture and supervision; it cannot isolate fine-tuning alone.',10,False),
        ('Experiments: no_context removes only the section description. lower_learning_rate changes only the learning rate from 2e-5 to 1e-5. Deltas around 0.005 may be seed noise and should not be treated as a reliable improvement without replication.',10,False)])
    report_page(pdf,'Error analysis: sections and score calibration',[
        (sections.round(4).to_string(index=False),9,True),
        (calibration.round(4).to_string(index=False),9,True),
        ('Section correlations are less stable with fewer examples. Domain vocabulary and scarce section examples may explain harder sections, but these explanations need further testing. Predictions above zero for true-zero pairs and below one for true-one pairs indicate compression toward the middle. Frequent intermediate labels and MSE can encourage this pattern.',10,False)])
    error_lines = []
    for r in worst.itertuples():
        error_lines += textwrap.wrap(f'{r.anchor} / {r.target} [{r.context}] true={r.score:.2f}, pred={r.pred:.3f}, error={r.absolute_error:.3f}; {r.inspection_group}',92)
        error_lines.append('')
    report_page(pdf,'Ten largest validation errors',[
        ('\n'.join(error_lines),9,True),
        ('Groups are lexical inspection aids, not confirmed causes. Check abbreviation expansion, phrase scope and technical synonymy in context. Do not call a label incorrect solely because the model disagrees.',10,False)])
    report_page(pdf,'Leaderboard, limitations and references',[
        (leaderboard.to_string(index=False) if len(leaderboard) else 'No leaderboard submission recorded. Public/private scores and screenshot require the real Kaggle late-submission run.',10,False),
        ('Validation, public and private scores can differ because the anchors, CPC-section mix and sample sizes differ. Repeated validation choices can overfit validation. Select by validation only, never by private leaderboard results. A single seed does not quantify uncertainty.',10,False),
        ('Sources: course project brief and starter (bk-anupam/generative_ai_shared, module3/labs/patent_phrase_matching); Kaggle U.S. Patent Phrase to Phrase Matching (2022); Aslanyan & Wetherbee (2022), arXiv:2208.01171; He et al. (2021), DeBERTaV3, arXiv:2111.09543; Reimers & Gurevych (2019), Sentence-BERT, arXiv:1908.10084; Hugging Face model cards microsoft/deberta-v3-small and sentence-transformers/all-MiniLM-L6-v2. Models retain their upstream licenses. No published competition solutions were copied.',10,False)])
print('Report:',OUT / 'patent_report_chirag.pdf')